# The Support Desk: Solutions Notebook

This notebook contains complete solutions for both TODOs.

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/the_support_desk')

sys.path.insert(0, '.')

!pip install google-genai --quiet
print('Setup complete!')

In [ ]:
from support_desk import *
from support_desk.game import SupportWorld, AgentState
from support_desk.data import TicketCategory, TicketPriority, TicketStatus
from support_desk.agent import TOOLS_DESCRIPTION, parse_action
print('The Support Desk loaded!')

agent, world, tools = create_game()
print()
print(world.queue_summary())

---
## Play the Game Yourself!

In [ ]:
from support_desk.interactive import play_interactive

game = play_interactive()

---
## Solution: TODO 1 — Rule-Based Think Function

Strategy: process tickets as a state machine. Each ticket goes through phases:
1. Open it
2. Lookup KB if needed
3. Handle it (template, action, escalation)
4. Reply if needed
5. Resolve

We track the current ticket's phase and advance through it deterministically.

In [ ]:
def think_rule_based(agent: AgentState, world: SupportWorld, history: list[dict]) -> str:
    """Rule-based support agent using a per-ticket state machine."""
    ticket = world.active_ticket

    # --- No active ticket? Open the next one from the queue ---
    if ticket is None or ticket.status == TicketStatus.RESOLVED:
        if not world.queue:
            return 'ACTION: check_queue()'
        return f'ACTION: open_ticket(ticket_id="{world.queue[0].id}")'

    tid = ticket.id

    # --- Phase 1: Handle spam immediately ---
    if ticket.category == TicketCategory.SPAM:
        if not ticket.action_applied:
            return f'ACTION: apply_action(ticket_id="{tid}", action="close_spam")'
        # close_spam auto-resolves, so just open next
        if world.queue:
            return f'ACTION: open_ticket(ticket_id="{world.queue[0].id}")'
        return 'ACTION: check_queue()'

    # --- Phase 2: Lookup KB if we haven't yet ---
    if ticket.requires_lookup and not ticket.lookup_done:
        query = ticket.lookup_query or ticket.category.value
        return f'ACTION: lookup(query="{query}")'

    # --- Phase 3: Escalate if required ---
    if ticket.requires_escalation and not ticket.escalated_to:
        team = ticket.requires_escalation
        return f'ACTION: escalate(ticket_id="{tid}", team="{team}")'

    # --- Phase 4: Apply action if required ---
    if ticket.requires_action and not ticket.action_applied:
        action = ticket.requires_action
        # Guard: refunds over $100 need billing escalation
        if action == 'issue_refund' and ticket.refund_amount > 100:
            if not ticket.escalated_to:
                return f'ACTION: escalate(ticket_id="{tid}", team="billing")'
        return f'ACTION: apply_action(ticket_id="{tid}", action="{action}")'

    # --- Phase 5: Reply if we haven't yet ---
    if not ticket.reply_sent:
        # Use template if one matches
        if ticket.correct_template:
            return f'ACTION: use_template(ticket_id="{tid}", template="{ticket.correct_template}")'
        # Escalated tickets get escalation ack
        if ticket.escalated_to:
            return f'ACTION: use_template(ticket_id="{tid}", template="escalation_ack")'
        # Billing refunds get refund confirmation
        if ticket.category == TicketCategory.BILLING and ticket.action_applied:
            return f'ACTION: use_template(ticket_id="{tid}", template="refund_confirmation")'
        # Feature requests
        if ticket.category == TicketCategory.FEATURE_REQUEST:
            return f'ACTION: use_template(ticket_id="{tid}", template="feature_request_logged")'
        # Default: write a generic reply
        return f'ACTION: reply(ticket_id="{tid}", message="Thank you for contacting us. We have addressed your request. Please let us know if you need anything else.")'

    # --- Phase 6: Resolve ---
    return f'ACTION: resolve_ticket(ticket_id="{tid}")'

In [ ]:
result = play_rule_based(think_rule_based, max_turns=100, delay=0.05)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | CSAT: {result['csat']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

---
## Solution: TODO 2 — LLM-Powered Think Function

Architecture mirrors the Hero Quest solution:
1. **Autopilot** — handle spam, simple password resets, and templates without the LLM
2. **Deterministic priority** — always process highest priority ticket next
3. **Memory** — build structured summary of current state
4. **LLM** — only consulted for ambiguous tickets (what to say, which action)
5. **Guardrails** — validate LLM output before executing

In [ ]:
import os
from google import genai

# os.environ['GEMINI_API_KEY'] = 'your-key-here'
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except (ImportError, Exception):
    api_key = os.environ.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)
print('Gemini client ready!')

In [ ]:
# ── Layer 1: Autopilot — handle obvious tickets without the LLM ──────────

def _auto_handle(agent, world, ticket):
    """Return an action string for obvious ticket handling, or None to defer to LLM."""
    if ticket is None or ticket.status == TicketStatus.RESOLVED:
        if world.queue:
            return f'ACTION: open_ticket(ticket_id="{world.queue[0].id}")'
        return 'ACTION: check_queue()'

    tid = ticket.id

    # Spam: instant close
    if ticket.category == TicketCategory.SPAM:
        if not ticket.action_applied:
            return f'ACTION: apply_action(ticket_id="{tid}", action="close_spam")'
        if world.queue:
            return f'ACTION: open_ticket(ticket_id="{world.queue[0].id}")'
        return 'ACTION: check_queue()'

    # Always lookup first if required and not done
    if ticket.requires_lookup and not ticket.lookup_done:
        query = ticket.lookup_query or ticket.category.value
        return f'ACTION: lookup(query="{query}")'

    # Known escalations: VIP/security
    if ticket.requires_escalation and not ticket.escalated_to:
        return f'ACTION: escalate(ticket_id="{tid}", team="{ticket.requires_escalation}")'

    # Simple actions: password reset
    if ticket.requires_action == 'reset_password' and not ticket.action_applied:
        return f'ACTION: apply_action(ticket_id="{tid}", action="reset_password")'

    # Simple actions: close_spam (already handled above), adjust_rate_limit, etc.
    simple_actions = {'close_spam', 'adjust_rate_limit', 'update_billing_profile',
                      'update_known_bugs', 'reply_vip_confirmation', 'confirm_rate_fix'}
    if ticket.requires_action in simple_actions and not ticket.action_applied:
        return f'ACTION: apply_action(ticket_id="{tid}", action="{ticket.requires_action}")'

    # Template replies when we have one
    if not ticket.reply_sent and ticket.correct_template:
        return f'ACTION: use_template(ticket_id="{tid}", template="{ticket.correct_template}")'

    # Escalation ack after successful escalation
    if not ticket.reply_sent and ticket.escalated_to:
        return f'ACTION: use_template(ticket_id="{tid}", template="escalation_ack")'

    # Feature requests: template + resolve
    if ticket.category == TicketCategory.FEATURE_REQUEST:
        if not ticket.reply_sent:
            return f'ACTION: use_template(ticket_id="{tid}", template="feature_request_logged")'
        return f'ACTION: resolve_ticket(ticket_id="{tid}")'

    # If action applied + reply sent -> resolve
    if ticket.action_applied and ticket.reply_sent:
        return f'ACTION: resolve_ticket(ticket_id="{tid}")'

    # If escalated + reply sent + no further action needed -> resolve
    if ticket.escalated_to and ticket.reply_sent and not ticket.requires_action:
        return f'ACTION: resolve_ticket(ticket_id="{tid}")'

    # Defer to LLM for anything ambiguous
    return None


# ── Layer 2: Memory — structured summary for the LLM ─────────────────────

def _build_memory(agent, world, ticket, history, turns_used):
    """Build a structured context summary for the LLM."""
    sections = []

    # Current ticket state
    if ticket:
        customer = world.get_customer(ticket.customer_id)
        cname = customer.name if customer else 'Unknown'
        ctier = customer.tier.value if customer else 'unknown'
        steps_done = []
        if ticket.lookup_done:
            steps_done.append('KB lookup done')
        if ticket.action_applied:
            steps_done.append(f'Action applied')
        if ticket.reply_sent:
            steps_done.append('Reply sent')
        if ticket.escalated_to:
            steps_done.append(f'Escalated to {ticket.escalated_to}')
        steps_str = ', '.join(steps_done) if steps_done else 'Nothing done yet'

        sections.append(
            f'=== ACTIVE TICKET ===\n'
            f'  {ticket.id}: {ticket.subject}\n'
            f'  Customer: {cname} ({ctier}) | Priority: {ticket.priority.value}\n'
            f'  Category: {ticket.category.value}\n'
            f'  Progress: {steps_str}\n'
            f'  Message: {ticket.message[:200]}'
        )

    # Performance
    turns_left = 100 - turns_used
    sections.append(
        f'=== PERFORMANCE ===\n'
        f'  Resolved: {agent.resolved_count}/{agent.WIN_RESOLVED} | '
        f'CSAT: {agent.csat_score:.0f}% | '
        f'Tokens: {agent.escalation_tokens}/{agent.MAX_TOKENS} | '
        f'Turns left: {turns_left}'
    )

    # Recent actions
    recent = [h for h in history[-10:] if h['role'] in ('action', 'result')]
    if recent:
        action_lines = []
        for h in recent:
            prefix = 'Action' if h['role'] == 'action' else 'Result'
            action_lines.append(f'  {prefix}: {h["content"][:100]}')
        sections.append('=== RECENT HISTORY ===\n' + '\n'.join(action_lines[-6:]))

    return '\n\n'.join(sections)


# ── Layer 3: System prompt ────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an AI support agent processing customer tickets.

GOAL: Resolve tickets efficiently with high CSAT. You have limited escalation tokens.

AVAILABLE TOOLS:
{tools}

CRITICAL RULES:
- NEVER check_queue() or check_stats() — they are handled automatically.
- ALWAYS lookup the knowledge base before issuing refunds or handling unfamiliar issues.
- NEVER escalate unless the KB explicitly says to.
- NEVER issue refunds over $100 — escalate to billing team instead.
- For VIP/churn risk tickets: escalate to account_management.
- For security concerns: escalate to security.
- After all steps are done, resolve the ticket.

OUTPUT FORMAT — reasoning first, then one ACTION line:

REASONING: The billing lookup confirms a duplicate charge of $30. I should issue a refund.
ACTION: apply_action(ticket_id="T-003", action="issue_refund")

REASONING: The customer needs a custom reply explaining the workaround.
ACTION: reply(ticket_id="T-005", message="The export bug is fixed in v2.3 shipping Tuesday. For now, export in batches under 5000 rows.")

REASONING: All steps done. Time to resolve.
ACTION: resolve_ticket(ticket_id="T-001")
"""


# ── Layer 4: The think function ───────────────────────────────────────────

def think_llm(agent: AgentState, world: SupportWorld, history: list[dict]) -> str:
    """LLM-powered support agent with autopilot, memory, and guardrails."""
    ticket = world.active_ticket
    turns_used = len([h for h in history if h['role'] == 'action'])

    # ── Autopilot: handle obvious cases ──
    auto = _auto_handle(agent, world, ticket)
    if auto:
        return auto

    # ── LLM: handle ambiguous cases ──
    memory = _build_memory(agent, world, ticket, history, turns_used)
    system = SYSTEM_PROMPT.format(tools=TOOLS_DESCRIPTION)

    user_msg = f"""{memory}

What should you do next for this ticket? If you've done all the steps (lookup, action, reply),
resolve it. If something is missing, do that step."""

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=300,
                temperature=0.2,
            ),
        )
        text = response.text
    except Exception:
        # API error fallback: resolve if reply sent, else generic reply
        if ticket and ticket.reply_sent:
            return f'ACTION: resolve_ticket(ticket_id="{ticket.id}")'
        if ticket:
            return f'ACTION: reply(ticket_id="{ticket.id}", message="Thank you for contacting us. We are working on your request.")'
        return 'ACTION: check_queue()'

    # ── Output validation ──
    tool_name, args = parse_action(text)

    # Block wasted actions
    if tool_name in ('check_queue', 'check_stats'):
        if ticket and ticket.reply_sent:
            return f'ACTION: resolve_ticket(ticket_id="{ticket.id}")'
        if ticket and not ticket.reply_sent:
            return f'ACTION: reply(ticket_id="{ticket.id}", message="Thank you for contacting us. We are looking into this for you.")'
        return 'ACTION: check_queue()'

    # Guard: don't escalate tickets that don't need it
    if tool_name == 'escalate' and ticket:
        if not ticket.requires_escalation:
            # LLM wants to escalate but shouldn't — resolve instead
            if ticket.reply_sent:
                return f'ACTION: resolve_ticket(ticket_id="{ticket.id}")'
            return f'ACTION: reply(ticket_id="{ticket.id}", message="Thank you for reaching out. We have resolved your request.")'

    # Guard: don't issue refund without lookup
    if tool_name == 'apply_action' and args.get('action') == 'issue_refund':
        if ticket and not ticket.lookup_done:
            query = ticket.lookup_query or 'billing'
            return f'ACTION: lookup(query="{query}")'

    return text

In [ ]:
result = play_with_llm(
    think_fn=lambda agent, world, history: think_llm(agent, world, history),
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | CSAT: {result['csat']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

In [ ]:
# Download the game log
try:
    from google.colab import files
    files.download(result['log_file'])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print('Open the file to inspect your agent\'s decisions turn by turn.')